# مترجم خودکار مانگا / مانهوا به فارسی

OCR → پاک‌سازی حباب → ترجمه با **Gemini / OpenAI / DeepSeek / …** → رندر فارسی

**ترتیب:** سلول‌ها را از بالا به پایین با Shift+Enter اجرا کنید.

قبل از شروع: `Runtime → Change runtime type → GPU (T4)` را انتخاب کنید (اختیاری ولی توصیه‌شده).


## 1) نصب پیش‌نیازها


In [ ]:
# ساخت فایل constraints
with open("constraints.txt", "w") as f:
    f.write("""numpy==1.26.4
opencv-python-headless==4.8.1.78
opencv-python==4.8.1.78
opencv-contrib-python==4.8.1.78
""")

!pip install -q --upgrade pip setuptools wheel
!pip install -q --no-cache-dir --constraint constraints.txt numpy==1.26.4
!pip install -q --no-cache-dir --constraint constraints.txt opencv-python-headless==4.8.1.78

!pip install -q --no-cache-dir --no-deps --constraint constraints.txt paddlepaddle==2.6.2 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip install -q --no-cache-dir --no-deps --constraint constraints.txt paddleocr==2.7.0.3

!pip install -q --no-cache-dir --constraint constraints.txt pymupdf
!pip install -q --no-cache-dir --constraint constraints.txt --ignore-installed attrdict cython fire lxml openpyxl pdf2docx premailer python-docx visualdl
!pip install -q --no-cache-dir --constraint constraints.txt Pillow pyclipper lmdb scikit-image shapely python-bidi arabic-reshaper rapidfuzz imageio matplotlib tqdm requests beautifulsoup4 decorator imgaug opt-einsum astor pyyaml simple-lama-inpainting

!pip install -q --no-cache-dir google-genai openai

print("✅ نصب پیش‌نیازها تمام شد.")


Telegram:
@Amir_wolf512
-------------------
حمایت مالی ( •̀ ω •́ )✧
Ton:UQBScvayaxagwTfRBhlLNaqw-sZuadlnBjSvn8OJz7XZJJzT
-------------------
TRX:TMmLTaCjaW1L2xWZmpR2EBeNyCawCzEkwa
-------------------


## 2) نوشتن اسکریپت اصلی مترجم
این سلول فایل `manga_translator.py` را از ریپو می‌گیرد (پشتیبانی از چند ارائه‌دهنده AI).


In [ ]:
!git clone --depth 1 https://github.com/amirwolf5122/Manga-AutoTranslate.git
!cp Manga-AutoTranslate/manga_translator.py .
!rm -rf Manga-AutoTranslate
print("✅ manga_translator.py آماده است.")


Telegram:
@Amir_wolf512


## 3) دانلود فونت فارسی (Vazirmatn)
برای اینکه متن فارسی درست نمایش داده بشه، یک فونت پشتیبان فارسی لازم داریم.


In [ ]:
import os
os.makedirs('fonts', exist_ok=True)
!wget -q -O fonts/Vazirmatn-Bold.ttf \
  https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf/Vazirmatn-Bold.ttf
print('فونت:', 'fonts/Vazirmatn-Bold.ttf')


## 4) انتخاب ارائه‌دهنده AI و کلید API

| Provider | کلید رایگان / لینک |
|----------|---------------------|
| **gemini** | [aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) |
| **openai** | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |
| **deepseek** | [platform.deepseek.com](https://platform.deepseek.com/api_keys) |
| **groq** | [console.groq.com/keys](https://console.groq.com/keys) |
| **xai** | [console.x.ai](https://console.x.ai/) |
| **openrouter** | [openrouter.ai/keys](https://openrouter.ai/keys) |
| **ollama** | لوکال — در Colab معمولاً در دسترس نیست |


In [ ]:
from getpass import getpass
import os

print("ارائه‌دهنده AI را انتخاب کنید:")
print(" 1) gemini")
print(" 2) openai")
print(" 3) deepseek")
print(" 4) groq")
print(" 5) xai")
print(" 6) openrouter")
print(" 7) together")
choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"
_map = {
    "1": "gemini", "2": "openai", "3": "deepseek",
    "4": "groq", "5": "xai", "6": "openrouter", "7": "together",
}
PROVIDER = _map.get(choice, "gemini")
print(f"→ provider = {PROVIDER}")

MODEL_NAME = input("مدل (Enter = پیش‌فرض provider): ").strip() or None
if MODEL_NAME:
    print(f"→ model = {MODEL_NAME}")

print("\nکلیدهای API را یکی‌یکی وارد کن (خالی بذار تا تموم بشه):")
keys = []
while True:
    k = getpass(f"کلید {len(keys)+1} (Enter = پایان): ").strip()
    if not k:
        break
    keys.append(k)

if not keys:
    raise SystemExit("حداقل یک کلید لازم است.")

API_KEYS = ",".join(keys)
os.environ["API_KEY"] = API_KEYS  # fallback عمومی
# env مخصوص provider
_env_map = {
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "groq": "GROQ_API_KEY",
    "xai": "XAI_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "together": "TOGETHER_API_KEY",
}
os.environ[_env_map.get(PROVIDER, "API_KEY")] = API_KEYS
print(f"✅ {len(keys)} کلید برای {PROVIDER} ثبت شد.")


زبان اصلی متن منبع رو انتخاب کنید (این مهمه؛ انتخاب اشتباه باعث می‌شه OCR متن رو درست استخراج نکنه):


In [ ]:
print("زبان اصلی متن منبع رو انتخاب کنید:")
print(" 1) en (انگلیسی - اکثر اسکنلیشن‌ها)")
print(" 2) ja en (ژاپنی خام)")
print(" 3) ko en (کره‌ای خام)")
print(" 4) دستی وارد کنید")
lang_choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"
if lang_choice == "1":
    OCR_LANG = "en"
elif lang_choice == "2":
    OCR_LANG = "ja en"
elif lang_choice == "3":
    OCR_LANG = "ko en"
elif lang_choice == "4":
    OCR_LANG = input("زبان OCR را وارد کنید: ").strip() or "en"
else:
    OCR_LANG = "en"
print("زبان OCR:", OCR_LANG)

print("\nترتیب خواندن:")
print(" 1) rtl (راست به چپ)")
print(" 2) ltr (چپ به راست)")
order_choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"
READING_ORDER = "ltr" if order_choice == "2" else "rtl"
print("ترتیب خواندن:", READING_ORDER)


## 5) ورودی رو بدید و ترجمه رو اجرا کنید

این سلول خودش تشخیص می‌ده که چی بهش دادید:
- اگه یک **لینک** (http/https) وارد کنید، تصاویر همون صفحه خودکار دانلود می‌شن.
- اگه Enter بزنید، می‌تونید فایل ZIP/PDF/تصویر آپلود کنید.


In [ ]:
import os
from google.colab import files

INPUT_DIR = 'input_pages'
os.makedirs(INPUT_DIR, exist_ok=True)

url = input('اگه لینک صفحه دارید وارد کنید (وگرنه Enter بزنید تا فایل آپلود شود): ').strip()
if url:
    INPUT_PATH = url
    print('ورودی (لینک):', INPUT_PATH)
else:
    print('فایل‌ها را انتخاب کنید (تصویر / ZIP / PDF)...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('هیچ فایلی آپلود نشد.')
    # اگر یک فایل بود همان را استفاده کن؛ وگرنه پوشه
    names = list(uploaded.keys())
    if len(names) == 1:
        INPUT_PATH = names[0]
    else:
        for name in names:
            os.rename(name, os.path.join(INPUT_DIR, name))
        INPUT_PATH = INPUT_DIR
    print('ورودی:', INPUT_PATH)


خروجی پیش‌فرض **PDF** است و نام فایل خودکار از روی لینک یا فایل ورودی ساخته می‌شود.

برای مانگای ژاپنی خام `--ocr-lang ja en`، برای مانهوای کره‌ای `--ocr-lang ko en` استفاده کنید.


In [ ]:
import os
import subprocess
import shlex

FONT = 'fonts/Vazirmatn-Bold.ttf'
OUT_SPEC = '.pdf'  # یا .html / .zip / نام کامل

cmd = [
    'python', 'manga_translator.py',
    '-i', INPUT_PATH,
    '-o', OUT_SPEC,
    '--font', FONT,
    '--ocr-lang', *OCR_LANG.split(),
    '--reading-order', READING_ORDER,
    '--provider', PROVIDER,
    '--api-key', API_KEYS,
]
if MODEL_NAME:
    cmd += ['--model', MODEL_NAME]

print('دستور:', ' '.join(shlex.quote(c) for c in cmd))
print()
ret = subprocess.call(cmd)
if ret != 0:
    raise SystemExit(f'اجرا با کد {ret} متوقف شد.')

# پیدا کردن آخرین خروجی
import glob
cands = sorted(glob.glob('*' + OUT_SPEC), key=os.path.getmtime, reverse=True)
if not cands and OUT_SPEC.startswith('.'):
    cands = sorted(glob.glob('*' + OUT_SPEC), key=os.path.getmtime, reverse=True)
output_path = cands[0] if cands else None
print('خروجی:', output_path)


## 6) دانلود خروجی‌ها


In [ ]:
from google.colab import files
import os
from IPython.display import display, IFrame
from PIL import Image as PILImage

if 'output_path' not in globals() or not output_path or not os.path.exists(output_path):
    print('خروجی پیدا نشد. اول سلول اجرا را بزنید.')
else:
    print('دانلود:', output_path)
    files.download(output_path)
    if output_path.lower().endswith('.pdf'):
        try:
            display(IFrame(output_path, width=800, height=600))
        except Exception:
            pass


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>